# CYBR 3570 — Crypto Lab 04
# Block Ciphers and Modes of Operation

**Today's Big Question:** If AES is strong, why can encryption still fail?

This notebook explores block ciphers, AES, ECB, CBC, CTR, padding, IVs, nonces, and common misuse patterns. The goal is not to implement AES from scratch. Instead, you will use established cryptographic libraries and observe how different modes behave.

> **Warning:** Some examples intentionally use insecure modes or unsafe patterns for educational purposes. Do not copy those examples into real systems.

## Learning Objectives

By the end of this lab, you should be able to:

1. Explain what a block cipher does.
2. Use AES through Python's `cryptography` library.
3. Demonstrate why ECB leaks repeated plaintext patterns.
4. Use CBC with PKCS#7 padding and random IVs.
5. Use CTR mode and explain why nonce reuse is dangerous.
6. Measure a simple avalanche effect.
7. Add educational block-cipher helpers to the course toolkit.

## Setup

Run the following cell. If the import fails, install the course dependencies in your environment.

```bash
pip install cryptography matplotlib
```

In [2]:
from os import urandom
from pathlib import Path
from collections import Counter
import textwrap

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

BLOCK_SIZE_BYTES = 16
KEY_SIZE_BYTES = 16

print("Setup complete.")

Setup complete.


## 1. Concept Review

Answer each question briefly in the Markdown cells below.

### Question 1
What is a block cipher? In your answer, use the terms **key**, **plaintext block**, and **ciphertext block**.

**Your answer:**

### Question 2
Why is a block cipher alone not enough to encrypt a message of arbitrary length?

**Your answer:**

### Question 3
What is the difference between a block cipher and a mode of operation?

**Your answer:**

mode of operations is the process used. (DES, AES) block cipher treat 

## 2. Helper Functions

We'll begin with a few helper functions for displaying data and splitting byte strings into blocks.

In [3]:
def blocks(data: bytes, block_size: int = BLOCK_SIZE_BYTES) -> list[bytes]:
    """Split bytes into block-sized chunks."""
    return [data[i:i + block_size] for i in range(0, len(data), block_size)]


def show_blocks(data: bytes, block_size: int = BLOCK_SIZE_BYTES) -> str:
    """Return blocks as space-separated hex strings."""
    return " ".join(block.hex() for block in blocks(data, block_size))


def hamming_distance(a: bytes, b: bytes) -> int:
    """Count differing bits between two byte strings of equal length."""
    if len(a) != len(b):
        raise ValueError("Inputs must have equal length")
    return sum((x ^ y).bit_count() for x, y in zip(a, b))

sample = b"A" * 32
print(show_blocks(sample))

41414141414141414141414141414141 41414141414141414141414141414141


## 3. AES on a Single Block

AES processes 16-byte blocks. The following example encrypts one block using AES in ECB mode. ECB is not safe for message encryption, but it is useful here because it exposes the raw block transformation clearly.

In [15]:
key = urandom(KEY_SIZE_BYTES)
plaintext_block = bytes([0x00] * BLOCK_SIZE_BYTES)

cipher = Cipher(algorithms.AES(key), modes.ECB())
encryptor = cipher.encryptor()
ciphertext_block = encryptor.update(plaintext_block) + encryptor.finalize()

decryptor = cipher.decryptor()
recovered_block = decryptor.update(ciphertext_block) + decryptor.finalize()

print(f"key:        {key.hex()}")
print(f"plaintext:  {plaintext_block.hex()}")
print(f"ciphertext: {ciphertext_block.hex()}")
print(f"recovered:  {recovered_block.hex()}")
print("Recovered correctly?", recovered_block == plaintext_block)

key:        64ed58c32a792c52170939a88e053f18
plaintext:  00000000000000000000000000000000
ciphertext: be1ef6dc098ff95b7336a2f7961ccfc3
recovered:  00000000000000000000000000000000
Recovered correctly? True


### Checkpoint

Run the previous cell multiple times. Why does the ciphertext change between runs even though the plaintext block is always all zeros?

**Your answer:**

the key is randomly generate each time its used. 

## 4. ECB Mode: Pattern Leakage

ECB encrypts each block independently. If two plaintext blocks are identical, their ciphertext blocks are identical.

This is why ECB leaks patterns.

In [16]:
key = urandom(KEY_SIZE_BYTES)
plaintext = b"CYBR3570_BLOCK!!" * 4  # 16-byte block repeated 4 times
print("Plaintext blocks:")
print(show_blocks(plaintext))

cipher = Cipher(algorithms.AES(key), modes.ECB())
encryptor = cipher.encryptor()
ciphertext = encryptor.update(plaintext) + encryptor.finalize()

print("\nCiphertext blocks:")
print(show_blocks(ciphertext))
print("\nUnique ciphertext blocks:", len(set(blocks(ciphertext))))

Plaintext blocks:
43594252333537305f424c4f434b2121 43594252333537305f424c4f434b2121 43594252333537305f424c4f434b2121 43594252333537305f424c4f434b2121

Ciphertext blocks:
7908011e91d8e5ae76dbe68bcc691f9f 7908011e91d8e5ae76dbe68bcc691f9f 7908011e91d8e5ae76dbe68bcc691f9f 7908011e91d8e5ae76dbe68bcc691f9f

Unique ciphertext blocks: 1


### Question 4
What does the previous output demonstrate about ECB mode?

**Your answer:**
well encrypted but they are all encrypted the exact same making it easier to break. 

## 5. ECB Pattern Experiment

Create a plaintext made of repeated blocks and a plaintext made of different blocks. Encrypt both using AES-ECB. Compare the number of unique ciphertext blocks.

In [17]:
# TODO: Try modifying these plaintext values.
repeated_plaintext = b"A" * 64
varied_plaintext = b"Block 01 is here" + b"Block 02 is here" + b"Block 03 is here" + b"Block 04 is here"

# Ensure both are block-aligned.
print(len(repeated_plaintext), len(varied_plaintext))

key = urandom(KEY_SIZE_BYTES)
cipher = Cipher(algorithms.AES(key), modes.ECB())

def encrypt_ecb(data: bytes, key: bytes) -> bytes:
    encryptor = Cipher(algorithms.AES(key), modes.ECB()).encryptor()
    return encryptor.update(data) + encryptor.finalize()

c1 = encrypt_ecb(repeated_plaintext, key)
c2 = encrypt_ecb(varied_plaintext, key)

print("Repeated plaintext unique ciphertext blocks:", len(set(blocks(c1))))
print("Varied plaintext unique ciphertext blocks:   ", len(set(blocks(c2))))

64 64
Repeated plaintext unique ciphertext blocks: 1
Varied plaintext unique ciphertext blocks:    4


## 6. CBC Mode with PKCS#7 Padding

CBC chains blocks together and uses an initialization vector (IV). It also requires padding when the plaintext length is not a multiple of the block size.

The `cryptography` library requires you to handle padding explicitly when using low-level cipher APIs.

In [18]:
def pkcs7_pad(data: bytes, block_size_bits: int = 128) -> bytes:
    padder = padding.PKCS7(block_size_bits).padder()
    return padder.update(data) + padder.finalize()


def pkcs7_unpad(data: bytes, block_size_bits: int = 128) -> bytes:
    unpadder = padding.PKCS7(block_size_bits).unpadder()
    return unpadder.update(data) + unpadder.finalize()


def encrypt_cbc(plaintext: bytes, key: bytes, iv: bytes) -> bytes:
    padded = pkcs7_pad(plaintext)
    encryptor = Cipher(algorithms.AES(key), modes.CBC(iv)).encryptor()
    return encryptor.update(padded) + encryptor.finalize()


def decrypt_cbc(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    decryptor = Cipher(algorithms.AES(key), modes.CBC(iv)).decryptor()
    padded = decryptor.update(ciphertext) + decryptor.finalize()
    return pkcs7_unpad(padded)

key = urandom(KEY_SIZE_BYTES)
iv = urandom(BLOCK_SIZE_BYTES)
message = b"Meet me at PKI 153 after class."

ciphertext = encrypt_cbc(message, key, iv)
recovered = decrypt_cbc(ciphertext, key, iv)

print("message:   ", message)
print("iv:        ", iv.hex())
print("ciphertext:", show_blocks(ciphertext))
print("recovered: ", recovered)

message:    b'Meet me at PKI 153 after class.'
iv:         1566c60f944ff13c3a3a7f222c0f9b16
ciphertext: ebacf41f845b1503f75db2bc6a2f64db 363e56e65565b491273d5ee56ea7ddb7
recovered:  b'Meet me at PKI 153 after class.'


### Question 5
Why does CBC need padding for some messages but CTR does not?

**Your answer:**
because the CBC is chained to the last value, it needs to be the same length
CTR uses a nonce counter so is not chained to the prior values.

## 7. CBC and Random IVs

Encrypt the same message twice with the same key but different IVs. The ciphertexts should differ.

In [19]:
key = urandom(KEY_SIZE_BYTES)
message = b"Same message, same key, different IV."

iv1 = urandom(BLOCK_SIZE_BYTES)
iv2 = urandom(BLOCK_SIZE_BYTES)

c1 = encrypt_cbc(message, key, iv1)
c2 = encrypt_cbc(message, key, iv2)

print("IV 1:", iv1.hex())
print("C1:  ", show_blocks(c1))
print("IV 2:", iv2.hex())
print("C2:  ", show_blocks(c2))
print("Ciphertexts equal?", c1 == c2)

IV 1: 5d47bec7f86a9f3bfeab19fa64d12a04
C1:   87a7237b434fffeacd00bc8589294872 5e48709a7b31558b18720ee0f2737852 b5259808203bcb0817e6c721d9e6e663
IV 2: 8c67453158c5463586cf6479b88fe941
C2:   28a3542d4eb2acf4f0b3fb9869e9b003 f88732c12562ac8de56b9757416cb8dc d294ba88f50393b3db69d6305f268b05
Ciphertexts equal? False


### Question 6
What security problem would occur if CBC always used the same IV with the same key?

**Your answer:**
allows you ro figure out the first encrypted values and work forward.

## 8. CTR Mode

CTR mode turns a block cipher into a stream-like construction. It encrypts a nonce/counter value to produce a keystream, then XORs that keystream with the plaintext.

CTR does not need padding because the keystream can be truncated to the length of the plaintext.

In [20]:
def encrypt_ctr(data: bytes, key: bytes, nonce: bytes) -> bytes:
    encryptor = Cipher(algorithms.AES(key), modes.CTR(nonce)).encryptor()
    return encryptor.update(data) + encryptor.finalize()

key = urandom(KEY_SIZE_BYTES)
nonce = urandom(BLOCK_SIZE_BYTES)
message = b"CTR can encrypt any number of bytes, not just full blocks."

ciphertext = encrypt_ctr(message, key, nonce)
recovered = encrypt_ctr(ciphertext, key, nonce)  # same operation reverses it

print("nonce:     ", nonce.hex())
print("ciphertext:", ciphertext.hex())
print("recovered: ", recovered)
print("Recovered correctly?", recovered == message)

nonce:      80e333f737a6ffa38eb9f23d8724b9f1
ciphertext: fa1d2434f8af15a369e5a27fa89f5e6558d14dc8e94350b947a7aaa7d355620bdcb073dc1bf034cdacb4d5acd1428b359ff93da8aec8849ac515
recovered:  b'CTR can encrypt any number of bytes, not just full blocks.'
Recovered correctly? True


## 9. CTR Nonce Reuse Demonstration

CTR mode becomes dangerous if the same key and nonce are reused. This is similar to reusing a one-time pad.

If:

```text
C1 = P1 XOR S
C2 = P2 XOR S
```

then:

```text
C1 XOR C2 = P1 XOR P2
```

The keystream cancels out.

In [21]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

key = urandom(KEY_SIZE_BYTES)
nonce = urandom(BLOCK_SIZE_BYTES)

p1 = b"Attack at dawn. Send reinforcements."
p2 = b"Defend at dusk. Hold the south gate."

# Make the messages equal length for this demonstration.
min_len = min(len(p1), len(p2))
p1 = p1[:min_len]
p2 = p2[:min_len]

c1 = encrypt_ctr(p1, key, nonce)
c2 = encrypt_ctr(p2, key, nonce)

print("C1 XOR C2:", xor_bytes(c1, c2).hex())
print("P1 XOR P2:", xor_bytes(p1, p2).hex())
print("Equal?", xor_bytes(c1, c2) == xor_bytes(p1, p2))

C1 XOR C2: 051112040d0f000000000014040500001b0a020000060d0c4e150007170d4d020f001600
P1 XOR P2: 051112040d0f000000000014040500001b0a020000060d0c4e150007170d4d020f001600
Equal? True


### Question 7
Why is nonce reuse in CTR mode catastrophic?

**Your answer:**
the difference between two messages is the same for both plain text and cipher text.

## 10. Avalanche Effect

A strong block cipher should exhibit diffusion: changing one input bit should change many output bits.

This experiment flips one bit in a plaintext block and measures how many ciphertext bits change.

In [22]:
key = urandom(KEY_SIZE_BYTES)
p1 = bytes([0x00] * BLOCK_SIZE_BYTES)
p2 = bytearray(p1)
p2[0] ^= 0x01  # flip one bit
p2 = bytes(p2)

c1 = encrypt_ecb(p1, key)
c2 = encrypt_ecb(p2, key)

distance = hamming_distance(c1, c2)

print("P1:", p1.hex())
print("P2:", p2.hex())
print("C1:", c1.hex())
print("C2:", c2.hex())
print("Changed ciphertext bits:", distance, "out of", len(c1) * 8)

P1: 00000000000000000000000000000000
P2: 01000000000000000000000000000000
C1: b3b0f99ea459b0aabbf723234437748b
C2: 44660359a0748749f5a282ad931a3ed0
Changed ciphertext bits: 71 out of 128


### Question 8
What does this experiment suggest about diffusion in AES?

**Your answer:**
small changes in the input have big impact on the output.

## 11. Toolkit Update

For this week, add a new module to your toolkit:

```text
crypto_toolkit/
    symmetric/
        block_helpers.py
```

The module should include:

- `blocks(data: bytes, block_size: int = 16) -> list[bytes]`
- `show_blocks(data: bytes, block_size: int = 16) -> str`
- `xor_bytes(a: bytes, b: bytes) -> bytes`
- `hamming_distance(a: bytes, b: bytes) -> int`
- `pkcs7_pad(data: bytes, block_size_bits: int = 128) -> bytes`
- `pkcs7_unpad(data: bytes, block_size_bits: int = 128) -> bytes`

You may use the helper functions from this notebook as a starting point.

In [ ]:
# Optional: create a starter file for your toolkit.
module_dir = Path("crypto_toolkit/symmetric")
module_dir.mkdir(parents=True, exist_ok=True)
(module_dir / "__init__.py").touch()

starter = '"""\nEducational block cipher helper functions for CYBR 3570.\n\nWARNING:\nThese helpers are for learning and demonstration. Do not use this module as a\ncomplete production encryption library.\n"""\n\nfrom cryptography.hazmat.primitives import padding\n\n\ndef blocks(data: bytes, block_size: int = 16) -> list[bytes]:\n    """Split bytes into block-sized chunks."""\n    return [data[i:i + block_size] for i in range(0, len(data), block_size)]\n\n\ndef show_blocks(data: bytes, block_size: int = 16) -> str:\n    """Return blocks as space-separated hex strings."""\n    return " ".join(block.hex() for block in blocks(data, block_size))\n\n\ndef xor_bytes(a: bytes, b: bytes) -> bytes:\n    """XOR two byte strings up to the shorter length."""\n    return bytes(x ^ y for x, y in zip(a, b))\n\n\ndef hamming_distance(a: bytes, b: bytes) -> int:\n    """Count differing bits between two byte strings of equal length."""\n    if len(a) != len(b):\n        raise ValueError("Inputs must have equal length")\n    return sum((x ^ y).bit_count() for x, y in zip(a, b))\n\n\ndef pkcs7_pad(data: bytes, block_size_bits: int = 128) -> bytes:\n    """Apply PKCS#7 padding."""\n    padder = padding.PKCS7(block_size_bits).padder()\n    return padder.update(data) + padder.finalize()\n\n\ndef pkcs7_unpad(data: bytes, block_size_bits: int = 128) -> bytes:\n    """Remove PKCS#7 padding."""\n    unpadder = padding.PKCS7(block_size_bits).unpadder()\n    return unpadder.update(data) + unpadder.finalize()\n'

path = module_dir / "block_helpers.py"
if not path.exists():
    path.write_text(starter)
    print(f"Created {path}")
else:
    print(f"{path} already exists; not overwritten.")

## 12. Reflection

Return to today's big question:

> If AES is strong, why can encryption still fail?

Write a short paragraph that refers to at least two of the following: ECB, CBC IVs, padding, CTR nonce reuse, implementation, or authenticated encryption.

**Your reflection:**

## Submission Checklist

Before submitting, confirm that you have:

- [ ] answered all concept questions;
- [ ] run every code cell;
- [ ] explained the ECB pattern leakage result;
- [ ] explained CBC IV behavior;
- [ ] explained CTR nonce reuse;
- [ ] completed the avalanche experiment;
- [ ] added or updated `crypto_toolkit/symmetric/block_helpers.py`;
- [ ] committed and pushed your work to GitHub;
- [ ] submitted your repository link or notebook in Canvas as directed.